<table>
  <tr>
    <td><div align="left"><font size="30">System analysis</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bdsim_logo.png" width="300" height="120" style="height:auto"></td>
  </tr>
</table>

**Analyze and linearize a block-diagram system.**

(c) Peter Corke 2026

In [ ]:
# BDSIM_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _bdsim_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'bdsim'
IMPORT_NAME = 'bdsim'
GIT_URL: str | None = 'https://github.com/petercorke/bdsim'
JL_PACKAGES = []
JL_NO_DEPS: list[str] = []
JL_EXTRAS: dict[str, list[str]] = {'control': ['control']}
COLAB_PACKAGES = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed(extras=['control'], min_version='1.5.0')


We start with the standard imports, construct a simulator object, and use that to construct
a block diagram with all the `bdsim` blocks included as methods.

In [ ]:
import bdsim

sim = bdsim.BDSim(animation=False, quiet=True)  # create simulator

bd = sim.blockdiagram()  # create an empty block diagram

The model is similar to the *bdsim* example `eg1.py` but with a `SQRT` block inserted between the error summing junction and the `LTI_SISO` dynamics, which in this case are second order ($\zeta=0.4, \omega_n=1$)

In [ ]:
# define the blocks
demand = bd.STEP(T=1, on=1.0, name="demand")
sum = bd.SUM("+-")
gain = bd.GAIN(10)
sqrt = bd.SQRT(name="sqrt")
plant = bd.LTI_SISO(1, [1, 0.8, 1], name="plant")
scope = bd.SCOPE(styles=["k", "r--"], loc="lower right")

# connect the blocks
bd.connect(demand, sum[0], scope[1])
bd.connect(plant, sum[1])
bd.connect(sum, gain)
bd.connect(gain, sqrt)
bd.connect(sqrt, plant)
bd.connect(plant, scope[0])

bd.compile()  # check the diagram
sim.report(bd)

# Simulate the non-linear system

Our system is built, and we will simulate it for a unit step input

In [ ]:
demand.on = 1.0  # ensure the step height is 1
out_1 = sim.run(bd, T=20, watch=[demand, sqrt, plant])  # simulate

And now, we will simulate it for a smaller step input, and the time response is now quite different. This very clearly shows the impact of the square-root non-linearity.

In [ ]:
demand.on = 0.1  # change the on value of the step input
out_0_1 = sim.run(bd, T=20, watch=[demand, sqrt, plant])  # simulate

# System in functional form

Non-linear continuous-time systems can be generally described by a pair of equations
$$
\begin{align*}
\dot{x} &= f(x, u, t) \\
y &= g(x, u, t)
\end{align*}
$$

The `.functions()` method can return these functions, and they wrap *bdsim*'s internal block-diagram evaluation code.  The inputs $u$ in the equations above are any source blocks like `CONSTANT`, `STEP`, `WAVEFORM`. They are effectively removed from the block diagram, and their values are provided by the `u` parameter to the functions.  Similarly, the outputs $y$ are the signals flowing to any sink blocks like `SCOPE` or `ANIMATION` and they are effectively removed from the block diagram when these functions are evaluated. 

In [ ]:
F = bd.functions(full=True)
print(type(F))
print(F)


The return value is a `SystemFunctions` object. We see that our system has 1 input, what was the `STEP` block, but has two outputs which are the two inputs to the `SCOPE` block.  We can fix that by selecting just the first of those outputs:

In [ ]:
F = bd.functions(outputs=[scope[0]], full=True)
print(F)

This is now a single-input single-output system with two states. The states belong to the `LTI_SISO` block which by default uses controller canonic form – a chain of integrators with the input coupled to the first integrator.

We can extract the functions from the `SystemFunctions` object

In [ ]:
f = F.f
g = F.g

print(type(f))

When the states are zero, and the input is zero

In [ ]:
print("f([0,0],0):", f(x=[0, 0], u=0))

we see that $\dot{x}$ is zero as expected.  For an input of $u=1$

In [ ]:
print("f([0,0],1):", f(x=[0, 0], u=1))
print("g([0,0],1):", g(x=[0,0], u=1))

we see that the first element of $\dot{x}$ is non-zero but the output is zero. For an input that is $10 \times$ smaller, it is $\sqrt{10}\times$ smaller.

In [ ]:
print("f([0,0], 0.1):", f(x=[0, 0], u=0.1))
print("g([0,0], 0.1):", g(x=[0,0], u=0.1))

Simulating the block diagram using `sim.run(bd, ...)` is effectively integrating the function $f(\cdots)$ over time.

We can compute $f(\cdots)$ and $g(\cdots)$ for the final state in our simulations above, which is approximately the equilibrium state

In [ ]:
print("f(x_e, 1):", f(x=out_1.x[-1,:], u=1))
print("g(x_e, 1):", g(x=out_1.x[-1,:], u=1))

and we see that $\dot{x}$ is close to zero (it should be exactly zero at equilibrium), and that the output value of 0.916 matches what is shown in the time domain plot above for $u=1$.

We can find the equilibrium state more precisely by numerically solving $f(x, u) = 0$ for $x$, with $u=1$, using a root finder

In [ ]:
from scipy.optimize import root
import numpy as np

ret = root(lambda x: f(x=x, u=1), [0, 0])
print(ret)

and the result shows the equilibrium state is

In [ ]:
x_e = ret.x
print("x_e:   ", x_e)
print("f(x_e):", f(x=x_e, u=1))
print("g(x_e):", g(x=x_e, u=1))

The functions $f(\cdots)$ and $g(\cdots)$ have vector inputs, and the elements of $x$, $y$ and $u$ are associated with different blocks in the diagram.  The order is deterministic, in the order in which a block was added to the diagram.  We can determine programmatically what the elements correspond to using properties of the `SystemFunctions` object 

In [ ]:
print("u:", F.uindex)
print("y:", F.yindex)
print("x:", F.xindex)

These properties return an `Index` object which itself has various properties

In [ ]:
print("u:", F.uindex.labels)
for name in F.xindex.names:
    print(name)

Note that in this context, the number in square brackets after `plant` is the state index not output port number.

Computing $f(\cdots)$ and $g(\cdots)$ each involves evaluating the entire block diagram and is computationally inefficient.  Instead we can use the `fg` property which returns a function that returns the values $f(\cdots)$ and $g(\cdots)$ at the same operating point defined by $x$ and $u$

In [ ]:
fg = F.fg
f_val, g_val = fg(x=x_e, u=1)
print("f(x_e):", f_val)
print("g(x_e):", g_val)

A simpler way to obtain the two functions is by

In [ ]:
f, g = bd.functions(outputs=[scope[0]])

# Linearization of the system

Around a particular operating point $(x_e, u_e, y_e)$ we can represent the block-diagram model as a linear time-invariant state-space system
$$
\begin{align*}
\dot{\bar{x}} &= \mathbf{A} \bar{x} + \mathbf{B} \bar{u} \\
\bar{y} &= \mathbf{C} \bar{x} + \mathbf{D} \bar{u} 
\end{align*}
$$
where $\bar{x} = x - x_e$, $\bar{u} = u - u_e$, $\bar{y} = y - y_e$.

The parameters $(\mathbf{A}, \mathbf{B}, \mathbf{C}, \mathbf{D})$ are obtained by numerical differentiation of the $f(\cdots)$ and $g(\cdots)$ functions introduced above
$$
\mathbf{A} = \frac{\partial f}{\partial x}, \mathbf{B} = \frac{\partial f}{\partial u}, \mathbf{C} = \frac{\partial g}{\partial x}, \mathbf{D} = \frac{\partial g}{\partial u}
$$

For the system built earlier, we obtain the linearization at $u=1$ by

In [ ]:
lin_1 = bd.linearize(x=x_e, u=1, inputs=["demand"], outputs=[scope[0]], full=True)
print(lin_1)

The last line shows the system poles, the eigenvalues of the $\mathbf{A}$ matrix, and we can see that these are stable but oscillatory. 

We can linearize about another equilibrium operating point, with $u=0.1$ – that's the second time-domain plot above.  This time we don't have the exact equilibrium states from minimization, so we will use the final states from the simulation.

In [ ]:
lin_0_1 = bd.linearize(x=out_0_1.x[-1,:], u=0.1, inputs=["demand"], outputs=[scope[0]], full=True)
print(lin_0_1)

There are a few of things worth commenting on here. Firstly, the $\mathbf{A}$ and $\mathbf{B}$ matrices are quite different.  The differences between the $\mathbf{A}$ matrices are reflected in the system poles which are less well damped for the $u=0.1$ case.

Secondly, the last line indicates that operating point given for linearization is not a proper equilibrium state, $f(\cdots)$ is not exactly zero, so the linearized model is actually
$$
\begin{align*}
\dot{\bar{x}} &= \mathbf{A} \bar{x} + \mathbf{B} \bar{u} + f_0\\
\bar{y} &= \mathbf{C} \bar{x} + \mathbf{D} \bar{u} 
\end{align*}
$$
Depending on what you want to do with the linearized model, this may or may not be a concern.

Thirdly, the square root non-linearity means the system has infinite gain at the operating point $x_e=[0,0], u_e=0$.  The numerical differentiation used to compute $\mathbf{A}$ and $\mathbf{B}$ will give incorrect results which are not flagged.

# Integration with `python-control`

[Python Control](https://github.com/python-control/python-control) is the go-to package for control design using Python.  It can be installed by

```
$ pip install control
```

If it's installed then we can export a Python Control state space object from *bdsim*

In [ ]:
sys = lin_1.ss()
print(sys)
print("poles:", sys.poles())

This is an ordinary `control.StateSpace` object, and its poles are the eigenvalues of $\mathbf{A}$ that we saw in the linearization above.

A good use of it is to check the linear model against the non-linear simulation we started with.  We start the non-linear system at the equilibrium state and step the demand from $u=1$ to $u=1+\Delta u$, then compare with the step response of the linear model, which is in terms of deviations from the operating point.  Python Control computes the linear response with `forced_response`.

In [ ]:
import control as ct
import matplotlib.pyplot as plt

y_e = g(x=x_e, u=1)[0]  # output at the equilibrium
t = np.linspace(0, 15, 301)

plant.x0 = x_e  # start at equilibrium
for du in [0.1, 1.0]:  # a small step, then a large step
    demand.off, demand.on = 1.0, 1.0 + du
    out = sim.run(bd, T=15, dt=0.05, watch=[plant])  # non-linear simulation
    plt.plot(out.t, out.y[:, 0] - y_e, label=f"non-linear, $\\Delta u$={du}")

    resp = ct.forced_response(sys, t, du * (t >= 1))  # linear model
    plt.plot(resp.time, resp.outputs, "--", label=f"linear, $\\Delta u$={du}")
    plt.title("Comparison of non-linear and linear responses")

plt.xlabel("time (s)")
plt.ylabel(r"$\bar{y}$")
plt.legend()
plt.grid(True)

# put the blocks back as they were
demand.off, demand.on = 0.0, 1.0
plant.x0 = [0.0, 0.0]

For the small step the linear model is a very good match for the non-linear system, but for the large step it overestimates the change in output, since the square-root non-linearity flattens the response away from the operating point.  This is the trade-off of linearization: a simple model that is accurate only close to the operating point.

# Where next?

With a state-space model in hand you can use anything in Python Control, such as pole placement or LQR design for a state-feedback regulator.  See the [cart-pole](cart-pole.ipynb) and [ball-beam](ball-beam.ipynb) notebooks for complete designs.